In [128]:
# --- import libraries ---
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional
import matplotlib.pyplot as plt
import seaborn as sns

PARTICIPANT_DATA_PATH = './participant_data/'

In [129]:
from utilities.preprocess import preprocess

index_event = "Borrow"
outcome_event = "Liquidated"
dataset_path = os.path.join(index_event, outcome_event)

train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))

train_df.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userBorrowAvgAmountUSD,userReserveMode,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount
0,1.00000,0.000,1.160624,0.207912,147302.0,8.669167,3.0,0,3.0,0,...,0.999892,DAI,64356.0,0.999892,0.0,0.693093,5.0,0.0,1.0,0.0
1,0.20000,1.999,1.199348,-0.207912,3185.0,8.832500,1.0,0,1.0,0,...,0.199924,DAI,3185.0,1.199815,0.0,0.182258,9.0,0.0,1.0,0.0
2,1.00000,1.999,5.535831,-0.207912,270.0,10.563333,1.0,0,1.0,0,...,1.000053,DAI,270.0,2.199868,0.0,0.693174,11.0,0.0,1.0,0.0
3,6.00000,1.999,5.535831,-0.207912,310.0,10.574444,1.0,0,1.0,0,...,3.000026,USDC,40.0,7.199868,0.0,1.791759,11.0,0.0,1.0,0.0
4,0.79299,1.999,7.933243,-0.207912,328.0,11.998333,1.0,0,1.0,0,...,0.793127,DAI,328.0,7.992995,0.0,0.583961,13.0,0.0,1.0,0.0


## Without Oversampling

In [130]:
X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)
lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)

In [131]:
status_0_count = lifelines_train_df["status"].value_counts()[0]
status_1_count = lifelines_train_df["status"].value_counts()[1]

print(f"No of status=0: {status_0_count}")
print(f"No of status=1: {status_1_count}")
print("-----------------------------------------")
print(f"% of status=0: {status_0_count/(status_0_count + status_1_count):.2f}")
print(f"% of status=1: {status_1_count/(status_0_count + status_1_count):.2f}")

No of status=0: 861864
No of status=1: 24044
-----------------------------------------
% of status=0: 0.97
% of status=1: 0.03


In [132]:
model = CoxPHFitter(penalizer=0.1)
model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')

<lifelines.CoxPHFitter: fitted with 885908 total observations, 861864 right-censored observations>

In [133]:
from lifelines.utils import concordance_index

train_risk = model.predict_partial_hazard(X_train)

# --- c-index using partial hazard as risk scores ---
c_index = concordance_index(
    lifelines_train_df["timeDiff"],       # observed times
    -train_risk,                          # negative because higher risk -> shorter survival
    lifelines_train_df["status"]          # event indicator
)
print(f"c-index: {c_index:.3f}")

c-index: 0.814


In [134]:
# model.print_summary()

In [135]:
summary_df = model.summary
summary_df.head()

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
userBorrowSum,-0.043944,0.957007,0.003270,-0.050353,-0.037536,0.950894,0.963160,0.0,-13.440306,3.509845e-41,134.387644
marketWithdrawAvgAmount,0.001245,1.001245,0.003100,-0.004830,0.007320,0.995181,1.007346,0.0,0.401535,6.880264e-01,0.539464
marketDepositAvgAmountUSD,0.012842,1.012924,0.003168,0.006633,0.019050,1.006655,1.019233,0.0,4.053685,5.041716e-05,14.275726
sinDayOfMonth,0.017731,1.017889,0.003019,0.011813,0.023649,1.011883,1.023931,0.0,5.872272,4.298617e-09,27.793480
userSecondsSinceFirstTransaction,0.026481,1.026835,0.003008,0.020585,0.032377,1.020798,1.032907,0.0,8.802642,1.336323e-18,59.376436


#### With Different p Thresholds

In [136]:
##-- select features with both low p-value and meaningful HR --##
    # keep covariates with HR < 0.8 or HR > 1.2, provided p < 0.05.
    # be cautious with HR ≈ 1.0 (like 0.98–1.02) -> effect is negligible

# coef -> regression coefficient (β)
# exp(coef) -> Hazard Ratio (HR) = exp(β) (row['p'] >= 0.001) and 

cols_to_keep = []
for index, row in summary_df.iterrows():
    if row['p'] <= 0.0005: 
        cols_to_keep.append(index)

print(len(cols_to_keep))

40


In [137]:
new_train_df = lifelines_train_df[cols_to_keep + ['timeDiff', 'status']]
new_train_df.shape

(885908, 42)

## With Oversampling Minority Class (status=1)

In [138]:
lifelines_train_df.head(2)

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount,timeDiff,status
0,-0.398558,-2.489321,-2.099994,0.289315,-0.785666,-0.570639,-0.396655,-0.156334,-0.892979,-0.402879,...,-0.079284,-1.733569,-0.398471,-1.254174,-1.577749,-2.370232,1.542022,-0.932965,75264503.0,0.0
1,-0.398558,-2.487828,-2.099970,-0.306477,-0.797257,-0.545239,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.130458,-1.733569,-0.398471,-1.382937,-1.577674,-2.370232,1.542022,-0.932965,75091115.0,0.0


In [139]:
status_0_count = lifelines_train_df["status"].value_counts()[0]
status_1_count = lifelines_train_df["status"].value_counts()[1]

print(f"No of status=0: {status_0_count}")
print(f"No of status=1: {status_1_count}")
print("-----------------------------------------")
print(f"% of status=0: {status_0_count/(status_0_count + status_1_count):.2f}")
print(f"% of status=1: {status_1_count/(status_0_count + status_1_count):.2f}")

No of status=0: 861864
No of status=1: 24044
-----------------------------------------
% of status=0: 0.97
% of status=1: 0.03


In [140]:
# cols_to_drop = ["id", "user", "pool", "Index Event", "Outcome Event", "type", "timestamp", "status"]

X_train = lifelines_train_df.drop(["status"], axis=1)
y_train = lifelines_train_df["status"]

# X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)
# lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)

In [141]:
from imblearn.over_sampling import ADASYN
from collections import Counter

print("Before:", Counter(y_train))

adasyn = ADASYN(sampling_strategy=0.1, random_state=42, n_neighbors=100)  # n_neighbors can be tuned
X_res, y_res = adasyn.fit_resample(X_train, y_train)

print("After:", Counter(y_res))

Before: Counter({0.0: 861864, 1.0: 24044})
After: Counter({0.0: 861864, 1.0: 87158})


In [142]:
lifelines_balanced_train_df = pd.concat([X_res, y_res.reset_index(drop=True)], axis=1)
lifelines_balanced_train_df.head()

,userBorrowSum,marketWithdrawAvgAmount,marketDepositAvgAmountUSD,sinDayOfMonth,userSecondsSinceFirstTransaction,timeOfDay,userActiveDaysWeekly,userLiquidationCount,userActiveDaysMonthly,userRepayCount,...,userSecondsSincePreviousTransaction,marketBorrowSumUSD,userRepaySum,logAmountUSD,marketDepositCount,marketRepayAvgAmountUSD,sinQuarter,marketLiquidationAvgAmount,timeDiff,status
0,-0.398558,-2.489321,-2.099994,0.289315,-0.785666,-0.570639,-0.396655,-0.156334,-0.892979,-0.402879,...,-0.079284,-1.733569,-0.398471,-1.254174,-1.577749,-2.370232,1.542022,-0.932965,75264503.0,0.0
1,-0.398558,-2.487828,-2.099970,-0.306477,-0.797257,-0.545239,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.130458,-1.733569,-0.398471,-1.382937,-1.577674,-2.370232,1.542022,-0.932965,75091115.0,0.0
2,-0.398558,-2.487828,-2.097217,-0.306477,-0.797492,-0.276081,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132896,-1.733569,-0.398471,-1.254154,-1.577637,-2.370232,1.542022,-0.932965,75084884.0,0.0
3,-0.398558,-2.487828,-2.097217,-0.306477,-0.797489,-0.274353,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.133089,-1.733569,-0.398471,-0.977241,-1.577637,-2.370232,1.542022,-0.932965,75084844.0,0.0
4,-0.398558,-2.487828,-2.095696,-0.306477,-0.797487,-0.052927,-1.190754,-0.156334,-1.072193,-0.402879,...,-0.132848,-1.733569,-0.398471,-1.281682,-1.577600,-2.370232,1.542022,-0.932965,75079718.0,0.0


In [143]:
status_0_count = lifelines_balanced_train_df["status"].value_counts()[0]
status_1_count = lifelines_balanced_train_df["status"].value_counts()[1]

print(f"No of status=0: {status_0_count}")
print(f"No of status=1: {status_1_count}")
print("-----------------------------------------")
print(f"% of status=0: {status_0_count/(status_0_count + status_1_count):.2f}")
print(f"% of status=1: {status_1_count/(status_0_count + status_1_count):.2f}")

No of status=0: 861864
No of status=1: 87158
-----------------------------------------
% of status=0: 0.91
% of status=1: 0.09


In [144]:
model = CoxPHFitter(penalizer=0.1)
model.fit(lifelines_balanced_train_df, duration_col='timeDiff', event_col='status')

<lifelines.CoxPHFitter: fitted with 949022 total observations, 861864 right-censored observations>

In [145]:
from lifelines.utils import concordance_index

train_risk = model.predict_partial_hazard(X_res)

# --- c-index using partial hazard as risk scores ---
c_index = concordance_index(
    lifelines_balanced_train_df["timeDiff"],       # observed times
    -train_risk,                                   # negative because higher risk -> shorter survival
    lifelines_balanced_train_df["status"]          # event indicator
)
print(f"c-index: {c_index:.3f}")

c-index: 0.807


## Lasoo 

In [146]:
# from lifelines.utils import concordance_index

# train_risk = model.predict_partial_hazard(X_train)

# # --- c-index using partial hazard as risk scores ---
# c_index = concordance_index(
#     df_balanced["timeDiff"],       # observed times
#     -train_risk,                   # negative because higher risk -> shorter survival
#     df_balanced["status"]          # event indicator
# )
# print(f"C-Index: {c_index:.3f}")

In [147]:
model.print_summary()

<lifelines.CoxPHFitter: fitted with 949022 total observations, 861864 right-censored observations>
             duration col = 'timeDiff'
                event col = 'status'
                penalizer = 0.1
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 949022
number of events observed = 87158
   partial log-likelihood = -1140848.65
         time fit was run = 2025-09-20 14:07:22 UTC

---
                                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                                            
userBorrowSum                       -0.08      0.92      0.00           -0.09           -0.07                0.92                0.93
marketWithdrawAvgAmount             -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
marketDepositAvgAmountUSD            0.02      1.02      0.00            0.02            0.03                1.02                1.03
sinDayOfMonth                        0.02      1.02      0.00            0.02            0.03                1.02                1.03
userSecondsSinceFirstTransaction     0.05      1.05      0.00            0.05            0.05                1.05                1.06
timeOfDay                            0.01      1.01      0.00            0.00            0.01                1.00                1.01
userActiveDaysWeekly                 0.13      1.14      0.00            0.13            0.14                1.13                1.15
userLiquidationCount                 0.20      1.22      0.00            0.20            0.20                1.22                1.22
userActiveDaysMonthly                0.09      1.10      0.00            0.09            0.10                1.09                1.11
userRepayCount                      -0.09      0.92      0.00           -0.09           -0.08                0.91                0.92
cosQuarter                          -0.00      1.00      0.00           -0.01            0.00                0.99                1.00
marketRepayAvgAmount                 0.01      1.01      0.00           -0.00            0.01                1.00                1.01
marketLiquidationSum                 0.01      1.01      0.00            0.00            0.02                1.00                1.02
userLiquidationSum                   0.03      1.03      0.00            0.03            0.04                1.03                1.04
sinDayOfQuarter                      0.02      1.02      0.00            0.01            0.02                1.01                1.02
marketLiquidationCount               0.02      1.02      0.00            0.01            0.02                1.01                1.02
userRepaySumUSD                     -0.08      0.92      0.00           -0.09           -0.07                0.92                0.93
userWithdrawCount                   -0.01      0.99      0.00           -0.02           -0.01                0.98                0.99
marketRepayCount                     0.00      1.00      0.00           -0.00            0.01                1.00                1.01
userLiquidationSumUSD               -0.07      0.93      0.00           -0.08           -0.07                0.93                0.94
marketWithdrawSum                    0.01      1.01      0.00           -0.00            0.01                1.00                1.01
userWithdrawSumUSD                  -0.03      0.97      0.00           -0.04           -0.02                0.96                0.98
marketBorrowSum                      0.00      1.00      0.00           -0.00            0.01                1.00                1.01
marketLiquidationAvgAmountUSD        0.00      1.00      0.00           -0.00            0.01                1.00                1.01
userDepositSumUSD                   -0.03      0.98      0.00           -0.

In [148]:
# from lassonet import LassoNetCoxRegressor

# model = LassoNetCoxRegressor(tie_approximation="breslow")

# X_np = X_train.values if hasattr(X_train, "values") else X_train
# y_np = y_train.values if hasattr(y_train, "values") else y_train

# oracle, order, wrong, paths, prob = model.stability_selection(X_np, y_np)

In [149]:
# order

In [150]:
order = [7, 69, 53, 45,  6, 19, 65, 73,  2, 36, 74, 26, 42, 43,  3, 52,  4, 67,
        17,  8, 37, 64, 38, 21, 39,  9,  1, 35, 31, 66, 28, 14, 30, 76, 13, 70,
        40, 33, 24, 23, 15, 49, 50, 55, 27, 34, 16,  0, 12, 61, 20, 18, 48, 29,
        77, 62, 44, 51, 47, 46, 72, 58, 41, 56, 10, 75, 63, 22, 11, 25, 71, 60,
        68, 32, 54, 59,  5, 57]

print(len(order))

78


In [151]:
columns = (lifelines_train_df.keys()).to_list()

In [152]:
# --- select the columns whose index is in order ---
selected_cols = ["timeDiff", "status"]
for i in order:
    selected_cols.append(columns[i])

print(len(selected_cols))

80


In [153]:
new_lifelines_train_df = lifelines_train_df[selected_cols]
new_lifelines_train_df.head()

,timeDiff,status,userLiquidationCount,userBorrowAvgAmountUSD,userBorrowCount,userLiquidationAvgAmount,userActiveDaysWeekly,userLiquidationSumUSD,userDepositCount,logAmountUSD,...,marketRepayAvgAmount,quarter,marketBorrowSumUSD,cosDayOfWeek,sinDayOfWeek,dayOfMonth,dayOfWeek,cosTimeOfDay,timeOfDay,sinTimeOfDay
0,75264503.0,0.0,-0.156334,-0.485221,-0.439298,-0.075548,-0.396655,-0.0872,-0.237205,-1.254174,...,-2.342678,-1.452898,-1.733569,0.978175,1.050891,-0.185892,-1.448052,-0.743173,-0.570639,1.148620
1,75091115.0,0.0,-0.156334,-0.485246,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.382937,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-0.788563,-0.545239,1.108079
2,75084884.0,0.0,-0.156334,-0.485221,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.254154,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.148216,-0.276081,0.575966
3,75084844.0,0.0,-0.156334,-0.485158,-0.439211,-0.075548,-1.190754,-0.0872,-0.239892,-0.977241,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.149720,-0.274353,0.572074
4,75079718.0,0.0,-0.156334,-0.485227,-0.439298,-0.075548,-1.190754,-0.0872,-0.239892,-1.281682,...,-2.342678,-1.452898,-1.733569,-1.183281,0.557057,0.043339,-0.420731,-1.246935,-0.052927,0.048505


In [154]:
model = CoxPHFitter(penalizer=0.1)
model.fit(new_lifelines_train_df, duration_col='timeDiff', event_col='status')

<lifelines.CoxPHFitter: fitted with 885908 total observations, 861864 right-censored observations>

In [155]:
# model.print_summary()
from lifelines.utils import concordance_index

train_risk = model.predict_partial_hazard(X_train)

# --- c-index using partial hazard as risk scores ---
c_index = concordance_index(
    lifelines_train_df["timeDiff"],       # observed times
    -train_risk,                          # negative because higher risk -> shorter survival
    lifelines_train_df["status"]          # event indicator
)
print(f"C-Index: {c_index:.3f}")

C-Index: 0.814


In [156]:
summary_df = model.summary
summary_df.head()

,coef,exp(coef),se(coef),coef lower 95%,coef upper 95%,exp(coef) lower 95%,exp(coef) upper 95%,cmp to,z,p,-log2(p)
covariate,,,,,,,,,,,
userLiquidationCount,0.178409,1.195314,0.002239,0.174020,0.182798,1.190080,1.200571,0.0,79.676038,0.000000e+00,inf
userBorrowAvgAmountUSD,-0.057935,0.943711,0.003265,-0.064335,-0.051536,0.937691,0.949770,0.0,-17.743118,1.948171e-70,231.572846
userBorrowCount,-0.047917,0.953213,0.003264,-0.054315,-0.041519,0.947134,0.959331,0.0,-14.678561,8.844718e-49,159.629660
userLiquidationAvgAmount,0.033026,1.033577,0.002519,0.028089,0.037963,1.028487,1.038692,0.0,13.111070,2.845553e-39,128.046487
userActiveDaysWeekly,0.062299,1.064281,0.003093,0.056237,0.068361,1.057848,1.070752,0.0,20.142015,3.162034e-90,297.312675
